# Data Exploration — QMSum + MeetingBank
Quick look at the normalized `Transcript` corpus before building the extraction pipeline.

In [1]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from pipeline.schema import Transcript

PROCESSED_ROOT = Path.cwd().parent / "data" / "processed"

transcripts = {}
for dataset_dir in sorted(PROCESSED_ROOT.iterdir()):
    transcripts[dataset_dir.name] = [
        Transcript.model_validate_json(p.read_text()) for p in sorted(dataset_dir.glob("*.json"))
    ]
    print(dataset_dir.name, len(transcripts[dataset_dir.name]))

meetingbank 6892


qmsum 232


In [2]:
import pandas as pd

rows = []
for dataset, ts in transcripts.items():
    for t in ts:
        rows.append({
            "dataset": dataset,
            "meeting_id": t.meeting_id,
            "split": t.split,
            "n_utterances": len(t.utterances),
            "n_speakers": len(t.participants),
            "n_chars": sum(len(u.text) for u in t.utterances),
            "has_summary": t.reference_summary is not None,
        })

df = pd.DataFrame(rows)
df.groupby("dataset")[["n_utterances", "n_speakers", "n_chars"]].describe().T

dataset               meetingbank          qmsum
n_utterances count    6892.000000     232.000000
             mean      233.672519     556.797414
             std       419.867337     328.116516
             min         4.000000      91.000000
             25%        40.750000     312.250000
             50%        90.000000     506.500000
             75%       233.250000     758.000000
             max      5720.000000    1832.000000
n_speakers   count    6892.000000     232.000000
             mean        0.000000       9.245690
             std         0.000000      18.499444
             min         0.000000       4.000000
             25%         0.000000       4.000000
             50%         0.000000       4.000000
             75%         0.000000       7.000000
             max         0.000000     105.000000
n_chars      count    6892.000000     232.000000
             mean    15964.376088   45493.745690
             std     29174.064001   26173.683100
             min       382.000000    4587.000000
             25%      2111.750000   26880.750000
             50%      5570.000000   38884.000000
             75%     16139.000000   59231.250000
             max    370418.000000  138380.000000

In [3]:
# Gold evaluation meetings selected for hand-labeling (Phase 4) - held out from prompt tuning
gold_ids = (Path.cwd().parent / "data" / "eval_gold" / "selected_meetings.txt").read_text().splitlines()
print(f"{len(gold_ids)} gold meetings:")
for gid in gold_ids:
    print(" ", gid)

20 gold meetings:
  qmsum_icsi_Bed008
  qmsum_icsi_Bed003
  qmsum_icsi_Bmr023
  qmsum_icsi_Bed016
  qmsum_icsi_Bro027
  qmsum_ami_ES2011d
  qmsum_ami_ES2011a
  qmsum_ami_ES2004d
  qmsum_ami_ES2004c
  qmsum_ami_TS3004b
  qmsum_committee_covid_4
  qmsum_committee_education_9
  qmsum_committee_education_4
  qmsum_committee_education_17
  qmsum_committee_education_13
  meetingbank_LongBeachCC_08202019_19-0784
  meetingbank_LongBeachCC_12082020_20-1195
  meetingbank_AlamedaCC_04212015_2015-1534
  meetingbank_LongBeachCC_10202015_15-1079
  meetingbank_DenverCityCouncil_07182022_22-0585


In [4]:
# Sample transcript from each dataset
for dataset, ts in transcripts.items():
    t = ts[0]
    print(f"=== {dataset}: {t.meeting_id} ===")
    print("Reference summary:", (t.reference_summary or "")[:300])
    print("First 5 utterances:")
    for u in t.utterances[:5]:
        print(f"  [{u.speaker or 'UNKNOWN'}] {u.text[:120]}")
    print()

=== meetingbank: meetingbank_AlamedaCC_01022019_2019-6216 ===
Reference summary: Public Hearing to Consider Adoption of Resolution Amending Master Fee Resolution No. 12191 to Revise Alameda Fire Department Transport Fees.  (Fire 3200)
First 5 utterances:
  [UNKNOWN] Good evening, mayor.
  [UNKNOWN] Council staff.
  [UNKNOWN] I'm Rick Sandberg, deputy fire chief for the department.
  [UNKNOWN] So I was here to answer questions.
  [UNKNOWN] I didn't know that you wanted a presentation, but I can tell you from the staff report that we are proposing to raise th

=== qmsum: qmsum_ami_ES2002a ===
Reference summary: This was the kick-off meeting for the project. First of all, Project Manager led each group member to know each other and introduced the project which was aiming to design remote control. Next, they discussed their favourite animal characteristics. Lastly, Project Manager mentioned how they worked o
First 5 utterances:
  [Project Manager] Okay Right {vocalsound} Um well this is th